# جلسه ۶: فراخوانی توابع و استفاده از ابزار

## اهداف
- درک نحوه فراخوانی توابع خارجی توسط LLMها
- تعریف اسکیمای ابزار برای API اوپن‌ای‌آی
- مدیریت پاسخ‌های فراخوانی تابع و بازگرداندن نتایج
- ساخت یک دستیار چند ابزاره

**مدت زمان:** ۴۰ دقیقه | **سطح:** متوسط

**چرا مهم است:** فراخوانی توابع به LLMها اجازه می‌دهد با دنیای واقعی تعامل داشته باشند — پایگاه‌های داده، APIها، ماشین‌حساب‌ها و هر کدی که می‌نویسید.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# بارگذاری متغیرهای محیطی از فایل .env
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"راه‌اندازی کامل شد! مدل: {MODEL}")

Setup complete! Model: gpt-5-mini


## ۱. فراخوانی توابع چگونه کار می‌کند

جریان کار:
1. **شما** توابع موجود (ابزارها) را با اسکیمای آنها تعریف می‌کنید
2. **LLM** تصمیم می‌گیرد کدام تابع را فراخوانی کند و آرگومان‌ها را تولید می‌کند
3. **شما** تابع را با آن آرگومان‌ها اجرا می‌کنید
4. **شما** نتیجه را به LLM برمی‌گردانید
5. **LLM** پاسخ نهایی به زبان طبیعی تولید می‌کند

```
پیام کاربر → LLM → "فراخوانی get_weather(city='Paris')" → کد شما → نتیجه → LLM → پاسخ نهایی
```

**نکته مهم:** LLM توابع را اجرا نمی‌کند — فقط تصمیم می‌گیرد کدام را با چه آرگومان‌هایی فراخوانی کند. شما آنها را اجرا می‌کنید.

## ۲. تعریف ابزارها

هر ابزار با موارد زیر تعریف می‌شود:
- `name`: نام تابع
- `description`: تابع چه کاری انجام می‌دهد (به LLM کمک می‌کند تصمیم بگیرد چه زمانی از آن استفاده کند)
- `parameters`: اسکیمای JSON که آرگومان‌های مورد انتظار را توصیف می‌کند

In [ ]:
# مرحله ۱: تعریف توابع پایتون واقعی
def get_weather(city):
    """شبیه‌سازی دریافت داده‌های آب‌وهوا (در برنامه واقعی، یک API آب‌وهوا فراخوانی می‌شود)."""
    weather_data = {
        "Paris": {"temp": 18, "condition": "Partly Cloudy", "humidity": 65},
        "Tokyo": {"temp": 24, "condition": "Sunny", "humidity": 45},
        "New York": {"temp": 12, "condition": "Rainy", "humidity": 80},
        "London": {"temp": 14, "condition": "Overcast", "humidity": 75},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown", "humidity": 50})
    return json.dumps({"city": city, **data})

def calculate(expression):
    """ارزیابی ایمن یک عبارت ریاضی."""
    # فقط عملیات ریاضی مجاز
    allowed_chars = set('0123456789+-*/.() ')
    if not all(c in allowed_chars for c in expression):
        return json.dumps({"error": "Invalid expression"})
    try:
        result = eval(expression)  # ایمن است چون ورودی را اعتبارسنجی کردیم
        return json.dumps({"expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})

# مرحله ۲: تعریف اسکیمای ابزارها برای API
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g., 'Paris' or 'Tokyo'"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Calculate a mathematical expression. Supports +, -, *, /, parentheses.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate, e.g., '(15 + 23) * 2'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print(f"تعداد {len(tools)} ابزار تعریف شد: {[t['function']['name'] for t in tools]}")

Defined 2 tools: ['get_weather', 'calculate']


## ۳. انجام یک فراخوانی تابع

وقتی LLM تصمیم به استفاده از یک ابزار می‌گیرد، پاسخ به جای محتوای معمولی شامل فیلد `tool_calls` است.

In [ ]:
# ارسال پیامی که نیاز به ابزار دارد
messages = [
    {"role": "system", "content": "You are a helpful assistant with access to weather and calculator tools."},
    {"role": "user", "content": "What's the weather like in Tokyo?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools
)

# بررسی اینکه آیا مدل می‌خواهد تابعی فراخوانی کند
message = response.choices[0].message
print(f"دلیل پایان: {response.choices[0].finish_reason}")
print(f"محتوا: {message.content}")
print(f"فراخوانی ابزار: {message.tool_calls}")

if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"\nتابع برای فراخوانی: {tool_call.function.name}")
    print(f"آرگومان‌ها: {tool_call.function.arguments}")

Finish reason: tool_calls
Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_P2GUZEiQwLnEGSvfo51demSx', function=Function(arguments='{"city":"Tokyo"}', name='get_weather'), type='function')]

Function to call: get_weather
Arguments: {"city":"Tokyo"}


## ۴. جریان کامل فراخوانی تابع

پس از اینکه LLM درخواست فراخوانی تابع می‌دهد، باید:
1. تابع را اجرا کنیم
2. نتیجه را به عنوان پیام `tool` برگردانیم
3. اجازه دهیم LLM پاسخ نهایی را تولید کند

In [ ]:
# نگاشت نام توابع به توابع واقعی پایتون
available_functions = {
    "get_weather": get_weather,
    "calculate": calculate
}

def run_with_tools(user_message):
    """جریان کامل فراخوانی تابع."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
        {"role": "user", "content": user_message}
    ]
    
    # اولین فراخوانی API — LLM ممکن است درخواست فراخوانی ابزار بدهد
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )
    
    message = response.choices[0].message
    
    # اگر فراخوانی ابزاری نبود، پاسخ مستقیم برگردان
    if not message.tool_calls:
        return message.content
    
    # پردازش هر فراخوانی ابزار
    messages.append(message)  # افزودن پیام فراخوانی ابزار دستیار
    
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        print(f"  فراخوانی: {function_name}({function_args})")
        
        # اجرای تابع
        function = available_functions[function_name]
        result = function(**function_args)
        
        print(f"  نتیجه: {result}")
        
        # افزودن نتیجه ابزار به پیام‌ها
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })
    
    # دومین فراخوانی API — LLM پاسخ نهایی را با استفاده از نتایج ابزار تولید می‌کند
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )
    
    return final_response.choices[0].message.content

# تست!
print("\n" + run_with_tools("What's the weather in Paris?"))

  Calling: get_weather({'city': 'Paris'})
  Result: {"city": "Paris", "temp": 18, "condition": "Partly Cloudy", "humidity": 65}



- City: Paris  
- Temperature: 18°C  
- Condition: Partly Cloudy  
- Humidity: 65%


In [ ]:
# تست با ماشین‌حساب
print(run_with_tools("What is (145 + 287) * 3?"))

  Calling: calculate({'expression': '(145 + 287) * 3'})
  Result: {"expression": "(145 + 287) * 3", "result": 1296}


The result is $1296$.


In [ ]:
# تست با سؤالی که نیاز به ابزار ندارد
print(run_with_tools("What is the capital of Germany?"))

The capital of Germany is Berlin.


## ۵. مدیریت فراخوانی‌های همزمان چند ابزار

LLM می‌تواند در یک پاسخ، چندین فراخوانی تابع را به صورت موازی درخواست کند.

In [ ]:
# این باید چندین فراخوانی ابزار همزمان ایجاد کند
result = run_with_tools("Compare the weather in Tokyo and London. Also, what is 72 * 1.8 + 32?")
print("\n" + result)

  Calling: get_weather({'city': 'Tokyo'})
  Result: {"city": "Tokyo", "temp": 24, "condition": "Sunny", "humidity": 45}
  Calling: get_weather({'city': 'London'})
  Result: {"city": "London", "temp": 14, "condition": "Overcast", "humidity": 75}
  Calling: calculate({'expression': '72 * 1.8 + 32'})
  Result: {"expression": "72 * 1.8 + 32", "result": 161.6}



- Tokyo: 24°C, Sunny, humidity 45%.
- London: 14°C, Overcast, humidity 75%.

The calculation 72 * 1.8 + 32 = 161.6.


## ۶. افزودن ابزار جدید

بیایید یک ابزار سوم اضافه کنیم تا نشان دهیم گسترش سیستم چقدر آسان است.

In [ ]:
# تابع جدید: دریافت تاریخ/ساعت فعلی
from datetime import datetime

def get_current_datetime(timezone="UTC"):
    """دریافت تاریخ و ساعت فعلی."""
    now = datetime.now()
    return json.dumps({
        "datetime": now.strftime("%Y-%m-%d %H:%M:%S"),
        "timezone": timezone
    })

# افزودن به لیست ابزارها
tools.append({
    "type": "function",
    "function": {
        "name": "get_current_datetime",
        "description": "Get the current date and time.",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone": {
                    "type": "string",
                    "description": "The timezone, e.g., 'UTC', 'EST'"
                }
            },
            "required": []
        }
    }
})

# به‌روزرسانی توابع موجود
available_functions["get_current_datetime"] = get_current_datetime

# تست ابزار جدید
result = run_with_tools("What time is it right now?")
print("\n" + result)

  Calling: get_current_datetime({'timezone': 'UTC'})
  Result: {"datetime": "2026-04-29 22:13:18", "timezone": "UTC"}



The current time is **2026-04-29 22:13:18 UTC**.


## تمرین: ابزار خودتان را بسازید

یک ابزار `search_products` اضافه کنید که در یک پایگاه‌داده ساده محصولات جستجو می‌کند.

In [ ]:
# پایگاه‌داده محصولات
products = [
    {"name": "Laptop Pro 15", "category": "electronics", "price": 1299, "rating": 4.5},
    {"name": "Wireless Mouse", "category": "electronics", "price": 29, "rating": 4.2},
    {"name": "Standing Desk", "category": "furniture", "price": 499, "rating": 4.7},
    {"name": "Noise-Cancelling Headphones", "category": "electronics", "price": 349, "rating": 4.8},
    {"name": "Ergonomic Chair", "category": "furniture", "price": 699, "rating": 4.6},
]

def search_products(category=None, max_price=None):
    """جستجوی محصولات بر اساس دسته‌بندی و/یا حداکثر قیمت."""
    results = products
    if category:
        results = [p for p in results if p["category"] == category]
    if max_price:
        results = [p for p in results if p["price"] <= max_price]
    return json.dumps(results)

# افزودن اسکیمای ابزار
tools.append({
    "type": "function",
    "function": {
        "name": "search_products",
        "description": "Search the product catalog by category and/or maximum price.",
        "parameters": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "description": "Product category: 'electronics' or 'furniture'",
                    "enum": ["electronics", "furniture"]
                },
                "max_price": {
                    "type": "number",
                    "description": "Maximum price in dollars"
                }
            },
            "required": []
        }
    }
})

available_functions["search_products"] = search_products

# تست
result = run_with_tools("Show me electronics under $100")
print("\n" + result)

print("\n" + "="*50 + "\n")

result = run_with_tools("What's the best-rated product you have?")
print("\n" + result)

  Calling: search_products({'category': 'electronics', 'max_price': 100})
  Result: [{"name": "Wireless Mouse", "category": "electronics", "price": 29, "rating": 4.2}]



- Wireless Mouse — $29 — ⭐ 4.2/5




  Calling: search_products({'category': None, 'max_price': None})
  Result: [{"name": "Laptop Pro 15", "category": "electronics", "price": 1299, "rating": 4.5}, {"name": "Wireless Mouse", "category": "electronics", "price": 29, "rating": 4.2}, {"name": "Standing Desk", "category": "furniture", "price": 499, "rating": 4.7}, {"name": "Noise-Cancelling Headphones", "category": "electronics", "price": 349, "rating": 4.8}, {"name": "Ergonomic Chair", "category": "furniture", "price": 699, "rating": 4.6}]



The top-rated product is:

- Name: Noise-Cancelling Headphones  
- Category: electronics  
- Price: $349  
- Rating: 4.8


## خلاصه

**جریان فراخوانی تابع:**
1. تعریف ابزارها با اسکیمای JSON
2. ارسال ابزارها به API همراه با پیام‌ها
3. LLM تصمیم می‌گیرد چه زمانی از ابزارها استفاده کند و آرگومان‌ها را تولید می‌کند
4. شما توابع را اجرا می‌کنید و نتایج را برمی‌گردانید
5. LLM پاسخ نهایی را تولید می‌کند

**نکات کلیدی:**
- LLM فقط *تصمیم می‌گیرد* کدام تابع فراخوانی شود — شما آن را اجرا می‌کنید
- توصیف خوب ابزارها به LLM کمک می‌کند درست انتخاب کند
- همیشه آرگومان‌های تابع را قبل از اجرا اعتبارسنجی کنید
- LLM می‌تواند چندین ابزار را به صورت موازی فراخوانی کند

**جلسه بعدی:** ساخت عامل‌های (Agent) خودمختار که از ابزارها در یک حلقه استفاده می‌کنند!